<a href="https://colab.research.google.com/github/laksamanaap/244107020021-machine-leaning-course-2026-/blob/main/JS09/JS09_Lab4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
!pip install annoy hnswlib faiss-cpu

import numpy as np
import time
from annoy import AnnoyIndex
import faiss
import hnswlib

n_data = 1_000_000
dim = 5
X = np.random.random((n_data, dim)).astype(np.float32)

query = np.random.random((1, dim)).astype(np.float32)
k = 10

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 29.4 MB/s  0:00:00


## Annoy

In [6]:
print("=== Annoy ===")
ann_index = AnnoyIndex(dim, 'euclidean')

start = time.time()
for i in range(n_data):
    ann_index.add_item(i, X[i])
ann_index.build(10)
build_time = time.time() - start

start = time.time()
neighbors = ann_index.get_nns_by_vector(query[0], k, include_distances=True)
query_time = time.time() - start

print("Build time:", build_time, "detik")
print("Query time:", query_time, "detik")
print("Neighbors:", neighbors[0][:5], "...")

=== Annoy ===
Build time: 26.91605281829834 detik
Query time: 0.0002186298370361328 detik
Neighbors: [593975, 733143, 553451, 702892, 147253] ...


## FAISS (IndexFlatL2)

In [7]:
print("=== FAISS (IndexFlatL2) ===")
faiss_index = faiss.IndexFlatL2(dim)

start = time.time()
faiss_index.add(X)
build_time = time.time() - start

start = time.time()
distances, indices = faiss_index.search(query, k)
query_time = time.time() - start

print("Build time:", build_time, "detik")
print("Query time:", query_time, "detik")
print("Neighbors:", indices[0][:5], "...")

=== FAISS (IndexFlatL2) ===
Build time: 0.016083240509033203 detik
Query time: 0.009442329406738281 detik
Neighbors: [593975 733143 553451 702892 147253] ...


## HNSW

In [8]:
print("=== HNSW (hnswlib) ===")
hnsw_index = hnswlib.Index(space='l2', dim=dim)

start = time.time()
hnsw_index.init_index(max_elements=n_data, ef_construction=200, M=16)
hnsw_index.add_items(X)
build_time = time.time() - start

hnsw_index.set_ef(50)

start = time.time()
labels, distances = hnsw_index.knn_query(query, k=k)
query_time = time.time() - start

print("Build time:", build_time, "detik")
print("Query time:", query_time, "detik")
print("Neighbors:", labels[0][:5], "...")

=== HNSW (hnswlib) ===
Build time: 184.09633779525757 detik
Query time: 0.0004189014434814453 detik
Neighbors: [593975 733143 553451 702892 147253] ...
